In [1]:
import os
# Disable Gradio analytics to prevent network timeout errors
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import gradio as gr

# ==========================================
# 1. Load pre-trained ResNet50 model
# ==========================================
print("Loading pre-trained model...")
# Use the officially recommended way to load weights
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
model.eval()  # Set model to evaluation mode

# The index for goldfish in ImageNet's 1000 categories is 1
GOLDFISH_CLASS_INDEX = 1

# ==========================================
# 2. Define image preprocessing pipeline
# ==========================================
# Must use the same preprocessing parameters as during model training
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ==========================================
# 3. Define prediction function
# ==========================================
def predict_goldfish_prob(image):
    if image is None:
        return "Please upload an image first!"

    # Convert image to RGB to avoid issues with PNG alpha channels
    image = image.convert('RGB')
    input_tensor = preprocess(image)
    input_batch = input_tensor.unsqueeze(0)

    # Use GPU if available, otherwise use CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    input_batch = input_batch.to(device)

    # Perform inference without calculating gradients
    with torch.no_grad():
        output = model(input_batch)

    # Calculate probabilities using Softmax
    probabilities = F.softmax(output[0], dim=0)
    goldfish_prob = probabilities[GOLDFISH_CLASS_INDEX].item() * 100

    # Scientific judgment: In 1000 classes, >10% probability indicates presence
    THRESHOLD = 10.0

    if goldfish_prob >= THRESHOLD:
        status = "✅ Result: Goldfish [DETECTED]"
    else:
        status = "❌ Result: Goldfish [NOT DETECTED]"

    # Format output string
    result_str = f"{status}\n"
    result_str += "=" * 30 + "\n"
    result_str += f"Original probability: {goldfish_prob:.2f}% (Threshold: {THRESHOLD}%)\n\n"

    # Additional info: Check top 3 predictions
    top3_prob, top3_catid = torch.topk(probabilities, 3)
    categories = weights.meta["categories"]

    result_str += "Top 3 most likely categories:\n"
    for i in range(top3_prob.size(0)):
        cat_name = categories[top3_catid[i]]
        prob = top3_prob[i].item() * 100
        result_str += f"Rank {i + 1}: {cat_name} ({prob:.2f}%)\n"

    return result_str

# ==========================================
# 4. Build Web UI with Gradio
# ==========================================
# Create interactive interface
interface = gr.Interface(
    fn=predict_goldfish_prob,  # Bound processing function
    inputs=gr.Image(type="pil", label="Upload Image"),  # Input component
    outputs=gr.Textbox(label="Detection Result", text_align="center"),  # Output component
    title="🐠 Goldfish Detector",
    description="Upload any image to check if it contains a goldfish.",
    theme="default"
)

# Start web service
if __name__ == "__main__":
    print("Starting web service...")
    # share=False runs locally on port 7860 by default
    interface.launch(share=False)

Loading pre-trained model...


C:\Users\dyz\anaconda3\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Starting web service...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Gradio Web UI Demonstration
![Gradio UI](ui_screenshot.png)